# Sıfırdan Dil Modeli · Bölüm 6
## Dikkat II: sorgu, anahtar, değer

Geçen bölümde en basit öz-dikkati yazdık: her kelime, vektörü kendisine ne kadar benziyorsa o kadar bakıyordu. Öğrenilen hiçbir şey yoktu. Bu defterde dikkate üç **öğrenilebilir matris** veriyoruz: `W_query`, `W_key`, `W_value`. Her kelimenin vektörü üç ayrı role çevrilir: **sorgu** (ne arıyorum?), **anahtar** (bende ne var?), **değer** (ne veriyorum?). Sonra skorları boyun kareköküne bölüyoruz (√d_k), hepsini bir sınıfa koyuyoruz, ve en sonda serinin modeli Çırak'ın eğitilmiş bir dikkat başını aynı sınıfa yüklüyoruz.

Adımlar Sebastian Raschka'nın *Build a Large Language Model (From Scratch)* kitabının 3.4. bölümünü izler. Kod, kitabın açık kaynak deposundaki kodun ([rasbt/LLMs-from-scratch](https://github.com/rasbt/LLMs-from-scratch), Apache-2.0) Türkçe bir cümle için uyarlamasıdır.

Hücreleri sırayla çalıştırın: hücreye tıklayın, **Shift + Enter**'a basın.

## 1. Cümle ve üç matris
Geçen bölümün cümlesi ve elle seçtiğimiz üç sayılık vektörler. Kelime vektörünün boyu `d_in` (3); sorgu, anahtar ve değerin boyu `d_out` (burada 2). `W_query`, `W_key` ve `W_value` 3 × 2'lik matrisler. `torch.rand` 0 ile 1 arasında rastgele sayılar verir; `torch.manual_seed(123)` her çalıştırmada aynı rastgele sayıları verir, sizin defterinizde de aynı sonuçlar çıksın diye. `nn.Parameter`, bunun öğrenilecek bir ağırlık olduğunu söyler; `requires_grad=False` ile şimdilik eğitmiyoruz (çıktılar sade kalsın diye).

In [ ]:
import torch

cumle = "Babam yalağın içinde kırılmış kaşağıyı gördü"
kelimeler = cumle.split()
inputs = torch.tensor(
  [[0.90, 0.10, 0.20],   # Babam
   [0.10, 0.40, 0.80],   # yalağın
   [0.10, 0.20, 0.70],   # içinde
   [0.20, 0.90, 0.30],   # kırılmış
   [0.10, 0.80, 0.30],   # kaşağıyı
   [0.70, 0.30, 0.20]]   # gördü
)
print(inputs.shape)

In [ ]:
x_5 = inputs[4]                  # kaşağıyı
d_in = inputs.shape[1]           # kelime vektörünün boyu: 3
d_out = 2                        # sorgu, anahtar ve değerin boyu

torch.manual_seed(123)
W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key   = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
print(W_query)

## 2. Sorgu, anahtar, değer
Aynı kelime vektörü üç matrisle çarpılır, üç ayrı vektör çıkar. `@` matris çarpımı: 3 sayılık vektör × 3 × 2'lik matris = 2 sayılık vektör.

In [ ]:
query_5 = x_5 @ W_query          # kaşağıyı ne arıyor?
key_5   = x_5 @ W_key            # kaşağıyı'da ne var?
value_5 = x_5 @ W_value          # kaşağıyı ne veriyor?
print("sorgu:  ", query_5)
print("anahtar:", key_5)
print("değer:  ", value_5)

Bütün kelimelerin anahtarları ve değerleri tek seferde: 6 × 3'lük girdi × 3 × 2'lik matris = 6 × 2.

In [ ]:
keys = inputs @ W_key
values = inputs @ W_value
print("keys.shape:", keys.shape)
print("values.shape:", values.shape)

## 3. Skorlar
Skor artık iki kelime vektörünün değil, **sorgu** ile **anahtarın** nokta çarpımı. Kaşağıyı'nın sorgusu ile kırılmış'ın anahtarı:

In [ ]:
keys_4 = keys[3]                       # kırılmış'ın anahtarı
attn_score_54 = query_5.dot(keys_4)    # kaşağıyı → kırılmış
print(attn_score_54)

Kaşağıyı'nın bütün skorları tek satırda: sorgu × anahtarların devriği (`.T`).

In [ ]:
attn_scores_5 = query_5 @ keys.T
print(attn_scores_5)

Sorgu ve anahtar ayrı matrislerden geldiği için skor tablosu artık simetrik değil: kaşağıyı'nın kırılmış'a bakışı, kırılmış'ın kaşağıyı'na bakışıyla aynı olmak zorunda değil.

In [ ]:
queries = inputs @ W_query
attn_scores = queries @ keys.T           # 6 × 6: her sorgu, her anahtarla
print("kaşağıyı → kırılmış:", attn_scores[4, 3])
print("kırılmış → kaşağıyı:", attn_scores[3, 4])

## 4. Ölçek: √d_k
Softmax skorların farkına bakar. Skorlar büyürse ne olur? Aynı skorları 10 ve 100 ile çarpıp softmax'a sokalım:

In [ ]:
for olcek in [1, 10, 100]:
    w = torch.softmax(attn_scores_5 * olcek, dim=-1)
    print(f"×{olcek:<4}", " ".join(f"{p:.2f}" for p in w))

Skorlar büyüdükçe softmax bütün ağırlığı tek kelimeye verir. Eğitimde bu kötüdür: öteki skorlar biraz değişse de ağırlıklar değişmez, gradyanlar sıfıra yaklaşır ve model öğrenemez.

Skorlar neden büyür? Nokta çarpım `d` tane çarpımı toplar; vektörler uzadıkça toplam da büyür. 1.000 rastgele sorgu ve anahtarla deneyelim (`torch.randn`: sıfır çevresinde rastgele sayılar; `std`: sayıların tipik büyüklüğü, *yayılım*):

In [ ]:
torch.manual_seed(123)
print("boy    yayılım   ÷√d")
for d in [2, 32, 512]:
    q = torch.randn(1000, d)             # 1000 rastgele sorgu
    k = torch.randn(1000, d)             # 1000 rastgele anahtar
    skorlar = (q * k).sum(dim=-1)        # 1000 nokta çarpım
    yayilim = skorlar.std()
    print(f"{d:<6} {yayilim:7.2f} {yayilim / d**0.5:6.2f}")

Yayılım boyun karekökü kadar büyüyor; skorları √d'ye bölünce boy ne olursa olsun yayılım 1 civarında kalıyor. Kitapta bu boyun adı `d_k`: anahtarların boyu.

In [ ]:
d_k = keys.shape[-1]
attn_weights_5 = torch.softmax(attn_scores_5 / d_k**0.5, dim=-1)
print(attn_weights_5)

## 5. Bağlam vektörü
Ağırlıklarla artık kelime vektörlerini değil, **değerleri** topluyoruz:

In [ ]:
context_vec_5 = attn_weights_5 @ values
print(context_vec_5)

## 6. Hepsi bir sınıfta: SelfAttention_v1 ve v2
`nn.Module`, PyTorch'ta her katmanın kalıbı: ağırlıklar `__init__`'te kurulur, hesap `forward`'da yapılır. `sa_v1(inputs)` yazınca `forward` çalışır. Aynı tohumla (123) aynı matrisler oluşur; beşinci satır, az önce adım adım bulduğumuz bağlam vektörü olmalı. `grad_fn`: PyTorch sonucun nasıl hesaplandığını hatırlıyor (eğitimde gradyan için).

In [ ]:
import torch.nn as nn

class SelfAttention_v1(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key   = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x):
        keys = x @ self.W_key
        queries = x @ self.W_query
        values = x @ self.W_value
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        context_vec = attn_weights @ values
        return context_vec

In [ ]:
torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in, d_out)
print(sa_v1(inputs))

İkinci sürüm, matrisler yerine PyTorch'un `nn.Linear` katmanını kullanır: matris çarpımını o yapar, başlangıç değerlerini daha iyi seçer, istenirse bir sabit terim (*bias*) ekler.

In [ ]:
class SelfAttention_v2(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        context_vec = attn_weights @ values
        return context_vec

In [ ]:
torch.manual_seed(789)
sa_v2 = SelfAttention_v2(d_in, d_out)
print(sa_v2(inputs))

Sonuçlar farklı, çünkü başlangıç ağırlıkları farklı. İki sınıf aynı hesabı mı yapıyor? v2'nin ağırlıklarını v1'e aktaralım (kitabın 3.1 alıştırması). `nn.Linear` ağırlığını devrik saklar, o yüzden `.T`:

In [ ]:
sa_v1.W_query = nn.Parameter(sa_v2.W_query.weight.T)
sa_v1.W_key   = nn.Parameter(sa_v2.W_key.weight.T)
sa_v1.W_value = nn.Parameter(sa_v2.W_value.weight.T)
print(sa_v1(inputs))

## 7. Çırak'ın eğitilmiş bir başı
Bizim matrislerimiz rastgele. Eğitilmiş bir modelde ne olur? Serinin modeli **Çırak**'ın üçüncü bloğundaki altıncı dikkat başını yükleyelim. Dosyada iki şey var: bu başın sorgu, anahtar ve değer ağırlıkları (256 → 32), ve cümlemizin, Çırak'ın ilk iki bloğundan geçtikten sonra bu başa gelen vektörleri (o blokları ileride kendimiz yazacağız). Üç dosyayı serinin deposundan indiriyoruz:

In [ ]:
import os
import urllib.request

DEPO = "https://raw.githubusercontent.com/kocahmet1/sifirdan-dil-modeli/main/"

def indir(path):
    dosya = os.path.basename(path)
    if not os.path.exists(dosya):
        urllib.request.urlretrieve(DEPO + path, dosya)
    print(dosya, os.path.getsize(dosya) // 1024, "KB")

for path in ["onceki_bolumler.py", "veri/cirak_tokenizer.json",
             "veri/cirak_bas.pt"]:
    indir(path)

Bölüm 2'de yazdığımız tokenizer'la cümleyi Çırak'ın parçalarına bölüyoruz; ısı haritası fonksiyonu da geçen bölümden:

In [ ]:
from onceki_bolumler import cirak_tokenizer, isi_haritasi

cirak = cirak_tokenizer()
ids = cirak.encode(cumle)
parcalar = [cirak.decode([i]) for i in ids]
print(len(ids), parcalar)

Aynı `SelfAttention_v2`, gerçek boyda: 256 sayılık girdi, 32 sayılık sorgu, anahtar ve değer, `bias` açık. `load_state_dict` Çırak'ın ağırlıklarını katmanlara yerleştirir.

In [ ]:
bas = torch.load("cirak_bas.pt")
X = bas["girdi"]                     # üçüncü bloğun dikkatine giren 15 vektör
sa_cirak = SelfAttention_v2(256, 32, qkv_bias=True)
sa_cirak.load_state_dict(bas["agirliklar"])
print(X.shape, "→", sa_cirak(X).shape)

Ağırlıkları çizmek için `forward`'un ilk satırlarını tekrar ediyoruz. `torch.no_grad()`: eğitmediğimiz için gradyan takibini kapatıyor.

In [ ]:
with torch.no_grad():
    queries = sa_cirak.W_query(X)
    keys = sa_cirak.W_key(X)
    weights = torch.softmax(queries @ keys.T / keys.shape[-1]**0.5, dim=-1)
isi_haritasi(weights, parcalar)

Kaşağıyı'nın dört parçası en çok nereye bakıyor?

In [ ]:
for i in range(10, 14):                   # kaşağıyı: ' k', 'aş', 'ağı', 'yı'
    j = weights[i].argmax()
    print(f"{parcalar[i]!r:8} → {parcalar[j]!r:8} {weights[i, j]:.2f}")

## Özet
Üç öğrenilebilir matris her kelimeyi üç role çevirir: **sorgu** (`W_query`), **anahtar** (`W_key`), **değer** (`W_value`). Skor, sorgu ile anahtarın nokta çarpımı; skorlar softmax'tan önce √d_k'ya bölünür ki boy büyüdükçe softmax tek bir kelimeye kilitlenmesin. Ağırlıklar değerleri toplar: bağlam vektörü. `SelfAttention_v1` ve `v2` aynı hesap; v2 `nn.Linear` kullanır.

Çırak'ın eğitilmiş başında desen benzerlikten gelmiyor: her parça bir öncekine bakıyor. Bunu eğitim öğretti. Üst üçgendeki değerlerin ise anlamı yok: Çırak eğitilirken orası hep kapalıydı. Sıradaki bölümde o üçgeni biz de kapatıyoruz.

**Kendin dene**
- `d_out`'u 4 yapıp `SelfAttention_v1`'i yeniden çalıştırın: çıktının şekli ne oluyor?
- Çırak'ın başında `/ keys.shape[-1]**0.5`'i silip ısı haritasını yeniden çizin: ne değişiyor?

**Sıradaki bölüm:** Dikkat III · maske ve çok başlı dikkat.

---
Kaynak: Sebastian Raschka, *Build a Large Language Model (From Scratch)*, Manning 2024, bölüm 3.4. Kod: [rasbt/LLMs-from-scratch](https://github.com/rasbt/LLMs-from-scratch) (Apache-2.0), Türkçe uyarlama: Sıfırdan Dil Modeli. Cümle: Ömer Seyfettin, "Kaşağı" (kamu malı). Çırak'ın ağırlıkları ve tokenizer'ı Türkçe Vikipedi metniyle (CC BY-SA 4.0) eğitildi.